# Figure 1 — Dataset Composition & Diversity

### Input files
| Variable | File | Content |
|---|---|---|
| `merged` | `data/merged_data.xlsx` | Project-level metadata (organisms, tissues, diseases) |
| `search` | `data/search_data 1.xlsx` | Raw-file-level metadata (instruments, enzymes, etc.) |
| `ACFM` | `data/final_acfm_stats.json` | All-confidence fragment map statistics |
| `LCMH` | `data/stats_lcfm_mcfm_hcfm.json` | Labelled-spectrum tiers (LCFM / MCFM / HCFM) |

---
### Structure
- **Part 1 — Dataset composition**: pipeline reduction, raw-file counts, spectra, tissue, disease
- **Part 2 — Technical**: acquisition mode, detectors, fragmentation, instruments
- **Part 3 — Biology**: kingdoms, organisms, enzymes, charge state, PTMs, quality metrics


In [ ]:
# ── Environment check — do not pip install from inside the notebook ───────────
# The environment is pinned by uv.lock, which is the source of truth: the
# constraints in pyproject.toml are floors, and the lock records exactly what
# resolved. Create it with ./setup_kernel.sh and select the resulting kernel.
#
# The pins matter because figure_3's zoom-region search ranks candidate windows
# whose separation scores frequently tie. Those rankings are total orders now,
# so the selection is reproducible across numpy versions -- this check guards
# against the rest of the stack drifting. Installing from a cell cannot fix a
# mismatch anyway: once numpy is imported, `pip install numpy==x` does not
# change the module already loaded in this kernel.
import importlib.metadata as _im
import pathlib as _pl
import warnings as _w

try:
    import tomllib as _tomllib
except ModuleNotFoundError:            # Python 3.10
    _tomllib = None

_root = next((p for p in (_pl.Path.cwd(), *_pl.Path.cwd().parents)
              if (p / 'uv.lock').is_file()), None)

# Packages that change what the figures show, or how they are laid out.
_WATCHED = ('numpy', 'pandas', 'pyarrow', 'scikit-learn', 'matplotlib', 'plotly')

if _root is None:
    _w.warn('uv.lock not found; skipping the environment check.')
elif _tomllib is None:
    _w.warn('tomllib needs Python 3.11+; skipping the environment check. '
            'setup_kernel.sh builds a 3.11 environment by default.')
else:
    # uv.lock can hold several resolution splits (requires-python spans 3.10-3.13),
    # so one package may be locked at more than one version. Any of them is valid.
    _locked = {}
    for _p in _tomllib.loads((_root / 'uv.lock').read_text()).get('package', []):
        _locked.setdefault(_p['name'], set()).add(_p['version'])

    _bad = []
    for _pkg in _WATCHED:
        _want = _locked.get(_pkg)
        if not _want:
            continue
        try:
            _got = _im.version(_pkg)
        except _im.PackageNotFoundError:
            _bad.append((_pkg, '/'.join(sorted(_want)), 'not installed'))
            continue
        if _got not in _want:
            _bad.append((_pkg, '/'.join(sorted(_want)), _got))

    if _bad:
        _msg = '\n'.join(f'  {p:16s} lock {w:12s} have {g}' for p, w, g in _bad)
        if any(p == 'numpy' for p, _, _ in _bad):
            raise RuntimeError(
                'Environment does not match uv.lock:\n' + _msg +
                '\n\nRun ./setup_kernel.sh and select the "InstaNovo-FM (figures)" kernel.')
        _w.warn('Environment differs from uv.lock:\n' + _msg)
    else:
        print(f'Environment matches uv.lock ({len(_WATCHED)} watched packages).')


In [ ]:
from pathlib import Path as SysPath

# ── Repo root, independent of where the kernel was started ───────────────────
# Path.cwd() only worked when Jupyter happened to be launched from the repo root;
# with the notebooks in notebooks/ that silently pointed everything one level down.
# Walk up instead, anchored on files that only exist at the root.
def _find_repo_root(start=None):
    _p = (start or SysPath.cwd()).resolve()
    for _cand in (_p, *_p.parents):
        if (_cand / 'config' / 'metadata_colors.json').is_file() and (_cand / 'data').is_dir():
            return _cand
    raise RuntimeError(
        'Repo root not found: expected an ancestor holding config/metadata_colors.json '
        f'and data/. Searched upward from {_p}.')

BASE_DIR = _find_repo_root()
MERGED_PATH = BASE_DIR / 'data' / 'merged_data.xlsx'
SEARCH_PATH = BASE_DIR / 'data' / 'search_data.xlsx'

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import numpy as np
import seaborn as sns
from matplotlib.path import Path
from scipy.stats import entropy
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')


In [ ]:
def set_publication_style():
    """
    Apply consistent, publication-ready matplotlib style.
    Uses Palatino (or nearest available serif alternative) and ticks theme.
    Color palette is coherent with Nature Methods guidelines (pastel, colorblind-safe).
    """
    sns.set_theme(style="ticks")
    plt.rcParams.update({
        # Font — Palatino with graceful fallbacks
        "font.family":      "serif",
        "font.serif":       ["Palatino", "Palatino Linotype", "TeX Gyre Pagella",
                             "Book Antiqua", "URW Palladio L", "DejaVu Serif"],
        "font.size":        14,
        "axes.titlesize":   16,
        "axes.labelsize":   15,
        "xtick.labelsize":  12,
        "ytick.labelsize":  12,
        # Axes & ticks
        "axes.linewidth":    1.5,
        "xtick.major.width": 1,
        "ytick.major.width": 1,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        # Figure
        "figure.dpi":       300,
        "figure.facecolor": "white",
        "axes.facecolor":   "white",
        # Legend
        "legend.fontsize":      13,
        "legend.frameon":       False,
        "legend.columnspacing": 1.5,
        # Grid
        "axes.grid":      True,
        "grid.alpha":     0.25,
        "grid.color":     "#CCCCCC",
        "grid.linewidth": 0.5,
    })


def get_figsize(width_ratio=1, total_width_inch=14.0):
    """
    Return (width, height) in inches scaled to a standard A4 column width.

    width_ratio:
        3 → full width  (~14.0"  × 4.67")
        2 → medium      (~9.33"  × 4.67")
        1 → square      (~4.67"  × 4.67")
    """
    ratios = {1: (1, 1), 2: (2, 1), 3: (3, 1)}
    w_mult, h_mult = ratios.get(width_ratio, (1, 1))
    actual_width  = (total_width_inch / 3) * w_mult
    actual_height = actual_width / w_mult
    return (actual_width, actual_height)

FIGURES_DIR = BASE_DIR / 'figures' / '1'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset each figure is computed on → filename suffix (single source of truth).
#    ACFM=all-confidence · lcfm/mcfm/hcfm=single tier · all_tiers=combined/compared
#    · catalog=search_data/merged_data XLSX metadata
#    · foundational=data/foundational.txt storage manifest.
FIG_DATASET = {
    # ── Main figure 1, in panel order ────────────────────────────────────────
    'fig_1b_tissue_distribution':          '_catalog',
    'fig_1c_disease_distribution':         '_catalog',
    'fig_1d_kingdom_radial_projects':      '_catalog',
    'fig_1e_tier_nesting_euler':           '_all_tiers',
    'fig_1f_instrument_family':            '_lcfm',
    'fig_1g_enzyme_donut':                 '_lcfm',
    'fig_1h_fragmentation_by_tier':        '_lcfm',
    'fig_1i_ptm_distribution':             '_lcfm',
    # ── Supplementary ────────────────────────────────────────────────────────
    'sup_fig_peptide_length':              '_lcfm',
    'sup_fig_pipeline_files':              '_foundational',
    'sup_fig_pipeline_volume':             '_foundational',
    'sup_fig_precursor_charge_by_tier':    '_all_tiers',
    'sup_fig_collision_energy_by_tier':    '_all_tiers',
    'sup_fig_hyperscore_by_tier':          '_all_tiers',
    'sup_fig_sankey_biological':           '_lcfm',
    'sup_fig_sankey_technical':            '_lcfm',
}

def save_fig(fig, name):
    suffix = FIG_DATASET.get(name)
    if suffix is None:
        print(f'  ⚠️  {name!r} not in FIG_DATASET — saved WITHOUT dataset label')
        suffix = ''
    path = str(FIGURES_DIR / f'{name}{suffix}') + '.svg'
    fig.savefig(path, bbox_inches='tight')
    print(f'  Saved → figures/1/{name}{suffix}.svg')

def fmt_n(n):
    if n >= 1e9: return f'{n/1e9:.1f}B'
    if n >= 1e6: return f'{n/1e6:.0f}M'
    if n >= 1e3: return f'{n/1e3:.0f}K'
    return str(int(round(n)))

set_publication_style()


In [ ]:
missing_files = [str(p) for p in [MERGED_PATH, SEARCH_PATH] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing input file(s): {missing_files}\nCurrent working directory: {BASE_DIR}"
    )

merged = pd.read_excel(MERGED_PATH)
search = pd.read_excel(SEARCH_PATH, sheet_name='search_data')

print(f'Working directory : {BASE_DIR}')
print(f'merged_data       : {merged.shape[0]:,} projects  | columns: {list(merged.columns)}')
print(f'search_data       : {search.shape[0]:,} raw files | columns: {list(search.columns)}')
print(f'Active font       : {plt.rcParams["font.family"]} → {plt.rcParams["font.serif"]}')

# ── Organism grouping (edit to add/rename groups) ──────────────────────────────
def org_group(s):
    s = str(s)
    if 'Homo sapiens' in s or 'Homo Sapiens' in s: return 'H. sapiens'
    if 'Arabidopsis' in s:   return 'A. thaliana'
    if 'Rattus' in s:        return 'R. norvegicus'
    if 'Mus musculus' in s:  return 'M. musculus'
    if 'Saccharomyces' in s: return 'S. cerevisiae'
    if 'Zea mays' in s:      return 'Z. mays'
    if 'Bacillus' in s:      return 'B. subtilis'
    return 'Other'

def inst_family(s):
    """Collapse a raw instrument model onto the families that have a colour in
    metadata_colors.json['instrument']. Used by every instrument panel so that
    a family keeps the same colour across the whole figure."""
    s = str(s).lower()
    if 'timstof' in s or 'tims' in s:                                  return 'TimsTOF'
    if 'astral'  in s:                                                 return 'Orbitrap Astral'
    if 'elite'   in s:                                                 return 'Orbitrap Elite'
    if 'velos'   in s:                                                 return 'Orbitrap Velos'
    if 'q exactive' in s:                                              return 'Q Exactive'
    if any(k in s for k in ('fusion', 'lumos', 'eclipse', 'exploris')): return 'Orbitrap Fusion'
    return 'Orbitrap (Other)'

def det_group(s):
    s = str(s)
    if 'Orbitrap|IonTrap' in s: return 'Orbitrap+IT'
    if 'Orbitrap' in s:         return 'Orbitrap'
    if 'Astral' in s:           return 'Astral'
    if 'TOF' in s:              return 'TOF'
    if 'Triple' in s:           return 'Triple Quad'
    return 'Ion Trap'

search['org_g'] = search['organism'].apply(org_group)
search['det_g'] = search['detector'].apply(det_group)

# ── Color palette — loaded from shared metadata_colors.json (single source of truth)
with open(BASE_DIR / 'config' / 'metadata_colors.json') as _cf:
    META_COLORS = json.load(_cf)

NEUTRAL = META_COLORS['_meta']['neutral_color']   # uniform fill for non-encoded fields
PALETTE = META_COLORS['palette']

ORG_COLORS = META_COLORS['organism']
ACQ_COLORS = META_COLORS['acquisition']      # DDA blue / DIA orange (Nature Methods convention)
DET_COLORS = META_COLORS['detector']

# Accent colors for summary stats card (figure-specific, not metadata categories)
C_LIGHT = '#EEF4FB'   # very light blue
C_TEAL  = '#2E78B8'   # medium blue
C_MID   = '#1A3F60'   # dark blue

# ── Derived tables used across multiple cells ──────────────────────────────────
total   = len(search)
orgs    = search['org_g'].value_counts()
acqs    = search['acquisition'].value_counts()
dets    = search['det_g'].value_counts()
org_acq = pd.crosstab(search['org_g'], search['acquisition'])
acq_det = pd.crosstab(search['acquisition'], search['det_g'])

print(f'\nData loaded — {total:,} raw files across {search["project"].nunique()} projects.')

# ── JSON data ─────────────────────────────────────────────────────────────────
try:
    import circlify
    HAS_CIRCLIFY = True
except ImportError:
    HAS_CIRCLIFY = False

with open(BASE_DIR / 'data' / 'final_acfm_stats.json')     as _f: ACFM = json.load(_f)
ACFM['tiers']['acfm_splits'] = ACFM['tiers']['acfm']   # final_acfm_stats.json nests ACFM under 'acfm'; alias to legacy key used below

# ACFM spectrum count reported in the manuscript. final_acfm_stats.json carries
# 1,802,272,487 from the stats job that produced it; a later audit counting both
# stores that hold the tier arrived at 1,625,276,573, and that is the figure the
# manuscript reports. The JSON is left as the job emitted it rather than edited,
# so use this constant wherever the count is displayed.
ACFM_SPECTRA = 1_625_276_573
with open(BASE_DIR / 'data' / 'stats_lcfm_mcfm_hcfm.json') as _f: LCMH = json.load(_f)

# ── Storage manifest (data/foundational.json) ──────────────────────────────────
# Authoritative on-disk file counts and byte totals per format. Note the file's
# 'size' column is just 'bytes' rendered in TiB/GiB — one quantity, two units — so
# only two independent measures exist here: file count and volume.
_FOUND_LABEL = {'raw': 'raw', 'mzml': 'mzML', 'acfm': 'ACFM',
                'lcfm': 'LCFM', 'mcfm': 'MCFM', 'hcfm': 'HCFM'}
# The file was renamed to .json but still holds the whitespace table, so accept
# either: real JSON {format: {files, bytes}} first, the table as a fallback.
_found_path = BASE_DIR / 'data' / 'foundational.json'
if not _found_path.exists():
    _found_path = BASE_DIR / 'data' / 'foundational.txt'
_found_raw = _found_path.read_text()

FOUNDATIONAL = {}


def _found_rows(raw):
    """Yield {format, files, bytes} rows from whichever shape the manifest is in.

    It has already been a whitespace table, then a flat {format: {...}} object,
    and is now {"formats": [...], "total": {...}}. Rather than track the current
    one, accept all three: the file is hand-maintained and will change again.
    """
    try:
        doc = json.loads(raw)
    except json.JSONDecodeError:
        for line in raw.splitlines():                       # whitespace table
            p = line.split()
            if len(p) >= 3 and p[0].lower() not in ('format', 'total'):
                yield {'format': p[0], 'files': p[1], 'bytes': p[2]}
        return
    if isinstance(doc, dict) and isinstance(doc.get('formats'), list):
        rows = doc['formats']                               # current shape
    elif isinstance(doc, list):
        rows = doc
    elif isinstance(doc, dict):
        rows = [{'format': k, **v} for k, v in doc.items() if isinstance(v, dict)]
    else:
        raise TypeError(f'Unrecognised manifest shape: {type(doc).__name__}')
    for r in rows:
        if str(r.get('format', '')).lower() not in ('format', 'total'):
            yield r


for _r in _found_rows(_found_raw):
    FOUNDATIONAL[_FOUND_LABEL.get(str(_r['format']).lower(), _r['format'])] = {
        'files': int(str(_r['files']).replace(',', '')),
        'bytes': int(str(_r['bytes']).replace(',', '')),
    }
if not FOUNDATIONAL:
    raise ValueError(f'No format rows parsed from {_found_path}')

PIPE_ORDER   = ['raw', 'mzML', 'ACFM', 'LCFM', 'MCFM', 'HCFM']
# Colour follows the entity: the four tiers keep their tier colours; only the two
# pre-model source formats are new, and they take neutral greys.
PIPE_COLORS  = {**META_COLORS['pipeline_source'], **META_COLORS['tier']}

def fmt_bytes(b):
    for _u, _d in (('TiB', 2**40), ('GiB', 2**30), ('MiB', 2**20)):
        if b >= _d:
            return f'{b/_d:,.1f} {_u}'
    return f'{b:,} B'

print('\nStorage manifest — ' + ', '.join(
    f'{k} {v["files"]:,} files / {fmt_bytes(v["bytes"])}' for k, v in FOUNDATIONAL.items()))

PASTEL = META_COLORS['pastel']
TIER_PAL = META_COLORS['tier']
TIER_ORDER   = ['LCFM', 'MCFM', 'HCFM']
TIER_COLORS3 = [TIER_PAL[t] for t in TIER_ORDER]
KINGDOM_COLORS_P = META_COLORS['kingdom']
FRAG_COLORS_P    = META_COLORS['fragmentation']
INST_COLORS_P    = META_COLORS['instrument']

print(f'Data loaded — {len(search):,} raw files across {search["project"].nunique()} projects')
print(f'ACFM : {ACFM_SPECTRA:,} spectra | {ACFM["tiers"]["acfm_splits"]["projects"]} projects')
print(f'LCFM : {LCMH["tiers"]["lcfm"]["spectra"]:,} spectra')
print(f'MCFM : {LCMH["tiers"]["mcfm"]["spectra"]:,} spectra')
print(f'HCFM : {LCMH["tiers"]["hcfm"]["spectra"]:,} spectra')


---
# Part 1 — Dataset Composition

Projects, raw files, spectra, tissue types, and disease categories.


In [ ]:
# ── Storage manifest figures, from data/foundational.json ────────────────────
set_publication_style()
plt.rcParams.update({'axes.grid': False})

# ── Figures B & C — storage manifest, from data/foundational.json ─────────────
# Bars, not a funnel: a funnel asserts monotone reduction, and the file counts are
# not monotone (mzML exceeds raw — one raw file can convert to several mzML — and
# MCFM/HCFM are equal). Volume *is* monotone but spans 17,576×, so it needs a log
# axis; a linear funnel would render the last two stages invisible.
_pipe_vals = {m: [FOUNDATIONAL[s][m] for s in PIPE_ORDER] for m in ('files', 'bytes')}
_pipe_cols = [PIPE_COLORS[s] for s in PIPE_ORDER]
_ys        = np.arange(len(PIPE_ORDER))[::-1]      # raw at the top

def _draw_pipe(metric, fmt, xlabel, log):
    _v = _pipe_vals[metric]
    fig, ax = plt.subplots(figsize=get_figsize(1))
    ax.barh(_ys, _v, height=0.62, color=_pipe_cols,
            edgecolor='white', linewidth=1.0, zorder=3)
    for _y, _x in zip(_ys, _v):
        # every bar carries its value: the pale ACFM fill needs a readable channel
        ax.text(_x * 1.06 if log else _x + max(_v) * 0.015, _y, fmt(_x),
                va='center', ha='left', fontsize=9, color='#333')
    if log:
        ax.set_xscale('log', base=10)
        ax.set_xlim(min(_v) / 4, max(_v) * 12)
    else:
        ax.set_xlim(0, max(_v) * 1.30)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_yticks(_ys)
    ax.set_yticklabels(PIPE_ORDER)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.grid(axis='y', alpha=0)
    sns.despine(ax=ax)
    fig.tight_layout()
    return fig

_pct = lambda v, ref: f'{v / ref * 100:.3g}% of raw'

fig_b = _draw_pipe('files', lambda v: f'{v:,}', 'Files', False)
save_fig(fig_b, 'sup_fig_pipeline_files')
plt.show()

fig_c = _draw_pipe(
    'bytes', lambda v: f'{fmt_bytes(v)}  ({_pct(v, _pipe_vals["bytes"][0])})',
    'Stored volume (log₁₀ scale)', True)
save_fig(fig_c, 'sup_fig_pipeline_volume')
plt.show()


In [ ]:

set_publication_style()
plt.rcParams.update({'axes.grid': False})

# ── Data Loading ───────────────────────────────────────────────────

_src = {
    'search_data\n(XLSX)':  {'raw_files': len(search), 'projects': search['project'].nunique(), 'spectra': None},
    'ACFM\n(JSON)':         {'raw_files': ACFM['tiers']['acfm_splits']['runs'], 'projects': ACFM['tiers']['acfm_splits']['projects'], 'spectra':  ACFM_SPECTRA},
    'LCFM\n(JSON)':         {'raw_files': LCMH['tiers']['lcfm']['runs'], 'projects': LCMH['tiers']['lcfm']['projects'], 'spectra':  LCMH['tiers']['lcfm']['spectra']},
    'MCFM\n(JSON)':         {'raw_files': LCMH['tiers']['mcfm']['runs'], 'projects': LCMH['tiers']['mcfm']['projects'], 'spectra':  LCMH['tiers']['mcfm']['spectra']},
    'HCFM\n(JSON)':         {'raw_files': LCMH['tiers']['hcfm']['runs'], 'projects': LCMH['tiers']['hcfm']['projects'], 'spectra':  LCMH['tiers']['hcfm']['spectra']},
}

_labels  = list(_src.keys())
_colors  = [PASTEL[0], PASTEL[1], TIER_PAL['LCFM'], TIER_PAL['MCFM'], TIER_PAL['HCFM']]


In [ ]:
# ── Tier nesting — area-proportional Euler diagram ───────────────────────────
# The confidence tiers are strictly nested (LCFM ⊃ MCFM ⊃ HCFM), so a Venn would be
# wrong: there are no spectra in MCFM that are not also in LCFM. All three circles
# are area-proportional and drawn on one scale — ACFM (1.8B, a 490× jump) would push
# HCFM below a pixel, so it is stated as context above the diagram rather than drawn.
# Fills come from META_COLORS['tier'], an ordinal ramp: one hue, monotone lightness,
# so the reader sees the tier order in the colour itself.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

def _fmt_p(n):
    """Finer-grained than the global fmt_n: 3.68M must not round to '4M'."""
    if n >= 1e9: return f'{n/1e9:.2f}B'
    if n >= 1e6: return f'{n/1e6:.1f}M'
    if n >= 1e3: return f'{n/1e3:.0f}K'
    return str(int(round(n)))

_n_acfm = ACFM_SPECTRA
_tiers  = [(t.upper(), LCMH['tiers'][t]['spectra']) for t in ('lcfm', 'mcfm', 'hcfm')]
_n_ref  = _tiers[0][1]                        # LCFM sets the scale
_radii  = [np.sqrt(v / _n_ref) for _, v in _tiers]

_INK = '#1A3F60'    # 4.83:1 on the LCFM fill, 12.6:1 on white

fig, ax = plt.subplots(figsize=get_figsize(1))
ax.set_xlim(-1.22, 1.98)
ax.set_ylim(-0.42, 2.48)
ax.set_aspect('equal')
ax.axis('off')

# ACFM stated as context — not drawn, so nothing on the canvas is off-scale
ax.text(-1.12, 2.34, f'from {_fmt_p(_n_acfm)} ACFM spectra', ha='left', va='center',
        fontsize=10.5, fontweight='bold', color=_INK)
ax.text(-1.12, 2.16, 'filtered into three nested confidence tiers', ha='left',
        va='center', fontsize=8.5, style='italic', color='#5A7A90')

# Nested tier circles — area proportional, tangent at the base
for (_name, _val), _r in zip(_tiers, _radii):
    ax.add_patch(plt.Circle((0.0, _r), _r, facecolor=TIER_PAL[_name],
                            edgecolor='white', linewidth=1.8, zorder=2))

_r_l, _r_m, _r_h = _radii
ax.text(0.0, _r_l + 0.54, f'LCFM\n{_fmt_p(_tiers[0][1])}', ha='center', va='center',
        fontsize=12.5, fontweight='bold', color=_INK, zorder=6, linespacing=1.35)
ax.text(0.0, _r_l + 0.21, f'{_tiers[0][1] / _n_acfm * 100:.1f}% of ACFM',
        ha='center', va='center', fontsize=9, color=_INK, alpha=0.8, zorder=6)

# MCFM / HCFM are too small to label in place — leader lines to the right
_ann = dict(arrowprops=dict(arrowstyle='-', color='#7A8B99', linewidth=0.9,
                            shrinkA=0, shrinkB=2),
            fontsize=9.5, va='center', ha='left', zorder=6, color=_INK)
ax.annotate(f'MCFM  {_fmt_p(_tiers[1][1])}\n{_tiers[1][1] / _n_ref * 100:.1f}% of LCFM',
            xy=(_r_m * 0.71, _r_m * 1.71), xytext=(0.68, 1.02), **_ann)
ax.annotate(f'HCFM  {_fmt_p(_tiers[2][1])}\n{_tiers[2][1] / _n_ref * 100:.1f}% of LCFM',
            xy=(_r_h * 0.98, _r_h * 1.02), xytext=(0.68, 0.12), **_ann)

ax.text(0.30, -0.32, 'circle area proportional to number of spectra',
        ha='center', va='center', fontsize=8.5, color='#888', zorder=6)

fig.tight_layout()
save_fig(fig, 'fig_1e_tier_nesting_euler')
plt.show()

# ── Summary Table ─────────────────────────────────────────────────────────────
print(f"{'Source':<22} {'Raw files':>12} {'Projects':>10} {'Spectra':>15}")
print("-" * 62)
for l, d in _src.items():
    s = fmt_n(d['spectra']) if d['spectra'] else 'N/A'
    print(f"{l.replace(chr(10),' '):<22} {d['raw_files']:>12,} {d['projects']:>10,} {s:>15}")


In [ ]:
# ── Tissue types ─────────────────────────────────────────────
FIG_SIZE     = get_figsize(1)
TOP_N_TISSUE = 12

tissue_counts = merged['tissue'].value_counts().dropna().head(TOP_N_TISSUE).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = NEUTRAL   # neutral fill — not color-encoded (same treatment as PTMs)
bars = ax.barh(tissue_counts.index, tissue_counts.values,
               color=colors, edgecolor='white', height=0.68)
for bar, val in zip(bars, tissue_counts.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=9, color='#444')
ax.set_xlabel('Projects', fontsize=10)
ax.set_xlim(0, tissue_counts.max() * 1.22)
ax.grid(axis='y', alpha=0)
ax.tick_params(axis='y', labelsize=9)
fig.tight_layout()
save_fig(fig, 'fig_1b_tissue_distribution')
plt.show()


In [ ]:
# ── Disease types ────────────────────────────────────────────
FIG_SIZE  = get_figsize(1)
TOP_N_DIS = 12

disease_counts = merged['disease'].value_counts().dropna().head(TOP_N_DIS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = NEUTRAL   # neutral fill — not color-encoded (same treatment as PTMs)
bars = ax.barh(disease_counts.index, disease_counts.values,
               color=colors, edgecolor='white', height=0.68)
for bar, val in zip(bars, disease_counts.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=9, color='#444')
ax.set_xlabel('Projects', fontsize=10)
ax.set_xlim(0, disease_counts.max() * 1.22)
ax.grid(axis='y', alpha=0)
ax.tick_params(axis='y', labelsize=9)
fig.tight_layout()
save_fig(fig, 'fig_1c_disease_distribution')
plt.show()


---
# Part 2 — Technical Characteristics

Acquisition mode, detector types, fragmentation methods, and instrument models.


In [ ]:
# ── Panel d — Instrument family donut ───────────────────────────────────────
set_publication_style()
plt.rcParams.update({'axes.grid': False})

_raw = LCMH['tiers']['lcfm']['instrument']['counts']
_fam = defaultdict(int)
for _nm, _cnt in _raw.items():
    _fam[inst_family(_nm)] += _cnt
_fam = dict(sorted(_fam.items(), key=lambda x: x[1], reverse=True))
_labels = list(_fam.keys())
_sizes  = list(_fam.values())
_colors = [INST_COLORS_P.get(l, PASTEL[7]) for l in _labels]

fig, ax = plt.subplots(figsize=get_figsize(1))
_wedges, _, _autotexts = ax.pie(
    _sizes, labels=None, colors=_colors,
    autopct=lambda p: f'{p:.1f}%' if p >= 4 else '',
    pctdistance=0.75,
    wedgeprops=dict(width=0.48, edgecolor='white', linewidth=1.2),
    startangle=90)
for _at in _autotexts:
    _at.set_fontsize(11); _at.set_color('black'); _at.set_fontweight('bold')
ax.text(0, 0, f'{fmt_n(sum(_sizes))}\nspectra', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#333')
ax.legend(_wedges, [f'{l}\n({fmt_n(s)})' for l, s in zip(_labels, _sizes)],
          loc='center left', bbox_to_anchor=(0.98, 0.5), fontsize=11, frameon=False)
ax.set_aspect('equal')
fig.tight_layout()
save_fig(fig, 'fig_1f_instrument_family')
plt.show()


In [ ]:
# ── Panel d — Instrument × Fragmentation sunburst ───────────────────────────
# Inner ring: instrument family (% of total raw files)
# Outer ring: fragmentation method within each family
# Each project's runs are distributed proportionally across (family, frag) pairs
# using per-project spectra fractions as weights.
try:
    import plotly.express as px
except ImportError:
    raise ImportError('plotly is missing: use the pinned kernel')
    import plotly.express as px

from collections import defaultdict

def _inst_family(name):
    nl = name.lower()
    if 'timstof' in nl or 'tims' in nl:                                   return 'TimsTOF'
    if 'astral'  in nl:                                                      return 'Orbitrap Astral'
    if 'q exactive' in nl or 'orbitrap q exactive' in nl:                  return 'Q Exactive'
    if any(k in nl for k in ('fusion', 'lumos', 'eclipse', 'exploris')):   return 'Orbitrap Fusion'
    return 'Orbitrap (Other)'

_cross      = defaultdict(float)
_total_runs = 0
for _tier in ['lcfm', 'mcfm', 'hcfm']:
    for _proj, _pd in LCMH['tiers'][_tier]['projects_detail'].items():
        _runs = _pd.get('runs', 0)
        _total_runs += _runs
        _ic = _pd.get('instrument',   {}).get('counts', {})
        _fc = _pd.get('fragmentation', {}).get('counts', {})
        _ti = sum(_ic.values()) or 1
        _tf = sum(_fc.values()) or 1
        _fam_c = defaultdict(float)
        for _nm, _cnt in _ic.items():
            _fam_c[_inst_family(_nm)] += _cnt
        for _fam, _fcnt in _fam_c.items():
            for _frag, _frcnt in _fc.items():
                _cross[(_fam, _frag)] += _runs * (_fcnt / _ti) * (_frcnt / _tf)

_fam_totals = defaultdict(float)
for (_fam, _frag), _val in _cross.items():
    _fam_totals[_fam] += _val

_rows = [dict(id='Total', label='Total', parent='', value=_total_runs)]
for _fam, _val in sorted(_fam_totals.items(), key=lambda x: -x[1]):
    _rows.append(dict(id=_fam, label=_fam, parent='Total', value=_val))
for (_fam, _frag), _val in _cross.items():
    if _val > 0:
        _rows.append(dict(id=f'{_fam}|{_frag}', label=_frag, parent=_fam, value=_val))

_df_sb = pd.DataFrame(_rows)

_color_map = {
    'Total':            '#D5D8DC',
    'Orbitrap Fusion':  INST_COLORS_P['Orbitrap Fusion'],
    'Q Exactive':       INST_COLORS_P['Q Exactive'],
    'Orbitrap Astral':  INST_COLORS_P['Orbitrap Astral'],
    'Orbitrap (Other)': INST_COLORS_P['Orbitrap (Other)'],
    'TimsTOF':          INST_COLORS_P['TimsTOF'],
    **FRAG_COLORS_P,
}

_fig = px.sunburst(
    _df_sb,
    ids='id',
    names='label',
    parents='parent',
    values='value',
    color='label',
    color_discrete_map=_color_map
)
_fig.update_traces(
    textinfo='label+percent entry',
    insidetextorientation='radial',
)
_fig.update_layout(
    font_family='Palatino',
    title_x=0.5,
    margin=dict(t=60, b=20, l=20, r=20),
    width=600, height=600,
)
_fig.show()
print(f'Total raw files (across 3 tiers): {_total_runs:,}')


In [ ]:
# ── Panel e — Fragmentation method breakdown (LCFM tier only) ────────────────
set_publication_style()

_frag_order = ['HCID', 'HCD', 'CID', 'ETD']
_ys = np.arange(len(_frag_order))

fig, ax = plt.subplots(figsize=get_figsize(1))
_raw   = LCMH['tiers']['lcfm']['fragmentation']['counts']
_total = sum(_raw.values())
_pcts  = [_raw.get(f, 0) / _total * 100 for f in _frag_order]
_bars  = ax.barh(_ys, _pcts, height=0.62,
                 color=[FRAG_COLORS_P.get(f, PALETTE[7]) for f in _frag_order],
                 edgecolor='white', linewidth=0.8)
for b, pct in zip(_bars, _pcts):
    if pct >= 0.05:
        ax.text(pct + 0.5, b.get_y() + b.get_height() / 2,
                f'{pct:.1f}%', va='center', ha='left', fontsize=10, color='black')

ax.set_yticks(_ys)
ax.set_yticklabels(_frag_order)
ax.set_xlabel('Spectra (%)')
ax.set_title('LCFM', loc='left', fontsize=12)
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, 'fig_1h_fragmentation_by_tier')
plt.show()


---
# Part 3 — Biological Diversity

Organism kingdoms, enzymes, charge states, PTMs, and quality metrics by confidence tier.


In [ ]:
# ── Kingdom radial bar chart — by PROJECT ────────────────────
# Same metric as the organism-annotation version, but each kingdom is counted
# once per project that contains >=1 organism of that kingdom (a multi-kingdom
# project is counted in each). Percentages are share of classified projects.
FIG_SIZE  = get_figsize(1)
SCALE     = 'sqrt'    # 'linear' | 'sqrt' | 'log'
R_INNER   = 0.28      # inner radius of bars
R_MAX     = 0.90      # outer radius at 100% scaled height
BAR_DEG   = 48        # angular width of each bar (degrees)
GAP_DEG   = 12        # gap between bars (degrees)

_CLASSIFY = {
    'Animalia': ['Homo ', 'Mus musculus', 'Rattus ', 'Bos ', 'Sus ', 'Ovis ', 'Canis ',
                 'Gallus ', 'Danio ', 'Drosophila', 'Caenorhabditis', 'Oryctolagus',
                 'Oncorhynchus', 'Oryzias', 'Apis ', 'Spongilla', 'Didelph', 'Serpentes',
                 'Hypsibius', 'Crisetulus', 'Capra'],
    'Plantae':  ['Arabidopsis', 'Zea mays', 'Oryza ', 'Glycine', 'Solanum', 'Triticum',
                 'Vitis ', 'Gossypium', 'Vigna', 'Porphyra'],
    'Fungi':    ['Saccharomyces', 'Candida ', 'Fusarium', 'Magnaporthe', 'Neurospora'],
    'Bacteria': ['Bacillus', 'Escherichia', 'Mycobacterium', 'Staphylococcus', 'Streptococcus',
                 'Pseudomonas', 'Clostridium', 'Bacteroides', 'Bifidobacterium', 'Prevotella',
                 'Ruminococcus', 'Roseburia', 'Veillonella', 'Fusobacterium', 'Synechocystis',
                 'Thauera', 'Coprococcus', 'Dorea', 'Eggerthella', 'Blautia'],
    'Archaea':  ['Halobacterium', 'Methanosarcina'],
    'Protista': ['Chlamydomonas', 'Chromera', 'Dictyostelium', 'Emiliania', 'Guillardia',
                 'Phaeodactylum', 'Plasmodium', 'Symbiodinium', 'Thalassiosira',
                 'Trypanosoma', 'Stentor'],
}
_KINGDOM_ORDER  = ['Animalia', 'Plantae', 'Fungi', 'Bacteria', 'Archaea', 'Protista']
_KINGDOM_COLORS = META_COLORS['kingdom']

def _classify(s):
    for k, kws in _CLASSIFY.items():
        if any(kw in str(s) for kw in kws):
            return k
    return 'Other'

_proj_counts     = {}
_proj_classified = set()
for _proj, _grp in search.groupby('project'):
    _kingdoms = set()
    for v in _grp['organism'].dropna():
        for o in str(v).split(';'):
            o = o.strip()
            if o and o.lower() not in ('', 'nan'):
                k = _classify(o)
                if k != 'Other':
                    _kingdoms.add(k)
    if _kingdoms:
        _proj_classified.add(_proj)
    for k in _kingdoms:
        _proj_counts[k] = _proj_counts.get(k, 0) + 1

_total   = len(_proj_classified)
_vals    = [_proj_counts.get(k, 0) for k in _KINGDOM_ORDER]
_pcts    = [v / _total * 100 for v in _vals]

_transform = {'linear': lambda x: x,
              'sqrt':   lambda x: x**0.5,
              'log':    lambda x: np.log1p(x)}[SCALE]
_scaled  = [_transform(v) for v in _vals]
_s_max   = max(_scaled)
_heights = [(s / _s_max) * (R_MAX - R_INNER) for s in _scaled]

# ── Draw ──────────────────────────────────────────────────────────────────────
set_publication_style()
plt.rcParams.update({'axes.grid': False})

fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.set_aspect('equal')
ax.axis('off')
_pad = R_MAX + 0.42
ax.set_xlim(-_pad, _pad); ax.set_ylim(-_pad, _pad)

n     = len(_KINGDOM_ORDER)
_step = BAR_DEG + GAP_DEG          # degrees per kingdom slot
_start = 90 + _step / 2            # first bar centered just past 12 o'clock

for i, (k, h, pct, val) in enumerate(zip(_KINGDOM_ORDER, _heights, _pcts, _vals)):
    _mid_deg = _start - i * _step           # center angle (clockwise from top)
    _theta   = np.radians(_mid_deg)
    _t1      = np.radians(_mid_deg - BAR_DEG / 2)
    _t2      = np.radians(_mid_deg + BAR_DEG / 2)
    _r_outer = R_INNER + h
    _col     = _KINGDOM_COLORS[k]

    # Bar wedge
    _n_arc = 40
    _ts    = np.linspace(_t1, _t2, _n_arc)
    _xs    = np.concatenate([R_INNER * np.cos(_ts),
                              _r_outer * np.cos(_ts[::-1])])
    _ys    = np.concatenate([R_INNER * np.sin(_ts),
                              _r_outer * np.sin(_ts[::-1])])
    ax.fill(_xs, _ys, color=_col, alpha=0.88, zorder=3)
    ax.plot(np.append(_xs, _xs[0]), np.append(_ys, _ys[0]),
            color='white', lw=1.2, zorder=4)

    # Radial tick at bar top
    _r_tick = _r_outer + 0.04
    ax.plot([_r_outer * np.cos(_theta), _r_tick * np.cos(_theta)],
            [_r_outer * np.sin(_theta), _r_tick * np.sin(_theta)],
            color=_col, lw=1.5, zorder=5)

    # Label
    _r_lbl = _r_tick + 0.06
    _xl, _yl = _r_lbl * np.cos(_theta), _r_lbl * np.sin(_theta)
    _ha = 'left'  if np.cos(_theta) >= 0 else 'right'
    _va = 'bottom' if np.sin(_theta) >= 0 else 'top'
    ax.text(_xl, _yl,
            f'{k}\n{val:,}  ({pct:.1f}%)',
            ha=_ha, va=_va, fontsize=8.5, color='#222',
            fontweight='bold' if pct > 5 else 'normal',
            linespacing=1.4, zorder=6)

# Inner circle
_ts_full = np.linspace(0, 2 * np.pi, 200)
ax.fill(R_INNER * np.cos(_ts_full), R_INNER * np.sin(_ts_full),
        color='white', zorder=2)
ax.plot(R_INNER * np.cos(_ts_full), R_INNER * np.sin(_ts_full),
        color='#CCCCCC', lw=1, zorder=2)

# Center label
ax.text(0, 0.06, f'{_total:,}', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#333', zorder=7)
ax.text(0, -0.09, 'projects', ha='center', va='top',
        fontsize=8, color='#888', linespacing=1.3, zorder=7)

# Scale footnote
_lbl = {'linear': 'linear scale', 'sqrt': '√-scale', 'log': 'log scale'}[SCALE]
ax.text(0, -_pad + 0.05, _lbl, ha='center', va='bottom',
        fontsize=8, color='#AAA', zorder=6)

fig.tight_layout()
save_fig(fig, 'fig_1d_kingdom_radial_projects')
plt.show()


In [ ]:
# ── Sankey diagrams — technical & biological feature flows ───────────────────
# Two stand-alone Sankeys built from the per-project LCFM breakdowns. Within each
# project the adjacent stages are combined under an independence assumption
# (joint ≈ product of the project's marginals), then summed across projects
# weighted by the project's run count — the same approximation as before.
SCALE_MODE = 'sqrt'      # 'linear' | 'sqrt' | 'log'
NODE_W     = 0.040
GAP        = 0.014
ALPHA_RIB  = 0.38
MIN_PCT    = 0.4         # nodes below this % of runs are merged into "Other"

_total = LCMH['tiers']['lcfm']['runs']
_PROJ  = LCMH['tiers']['lcfm']['projects_detail']

# ── Category mappers ─────────────────────────────────────────────────────────
_inst_fam = inst_family      # shared mapper defined in the setup cell

def _detector_g(nm):
    nl = str(nm).lower()
    if 'astral' in nl:                                       return 'Astral'
    if 'orbitrap' in nl and ('iontrap' in nl or '|' in nl):  return 'Orbitrap+IT'
    if 'orbitrap' in nl:                                     return 'Orbitrap'
    if 'tof' in nl:                                          return 'TOF'
    if 'iontrap' in nl or 'ion trap' in nl:                  return 'Ion Trap'
    return 'Other'

def _ident(x):
    return str(x)

def _enzyme_g(e):
    return str(e).title()

def _charge_g(z):
    try:
        zi = int(float(z))
    except (TypeError, ValueError):
        return None
    if zi <= 0:
        return None
    return str(zi) if zi <= 4 else '5+'

_KINGDOM_MAP = {
    'Animalia': ['homo', 'mus ', 'rattus', 'gallus', 'bos taurus', 'canis', 'capra',
                 'ovis', 'sus scrofa', 'danio', 'caenorhabditis', 'drosophila',
                 'apis mellifera', 'oryctolagus', 'oncorhynchus', 'oryzias',
                 'hypsibius', 'cricetulus', 'crisetulus', 'spongilla', 'didelph', 'serpentes'],
    'Plantae':  ['arabidopsis', 'zea mays', 'oryza', 'glycine max', 'triticum',
                 'solanum', 'vitis', 'gossypium', 'vigna', 'porphyra'],
    'Fungi':    ['saccharomyces', 'candida', 'fusarium', 'magnaporthe', 'neurospora'],
    'Bacteria': ['bacillus', 'bacteriodes', 'bacteroides', 'bifidobacterium', 'blautia',
                 'clostridium', 'coprococcus', 'dorea', 'eggerthella', 'escherichia',
                 'fusobacterium', 'mycobacterium', 'mycoplasma', 'parabacteriodes',
                 'parabacteroides', 'prevotella', 'roseburia', 'ruminococcus',
                 'staphylococcus', 'streptococcus', 'thauera', 'pseudomonas',
                 'veillonella', 'clam bacteria', 'synechocystis'],
    'Archaea':  ['halobacterium', 'methanosarcina'],
    'Protista': ['chlamydomonas', 'chromera', 'dictyostelium', 'emiliania', 'guillardia',
                 'phaeodactylum', 'plasmodium', 'symbiodinium', 'thalassiosira',
                 'trypanosoma', 'stentor'],
}
def _to_kingdom(s):
    first = str(s).lower().split(';')[0].strip()
    for _k, _kws in _KINGDOM_MAP.items():
        if any(_kw in first for _kw in _kws):
            return _k
    return 'Other'

# ── Stage colour palettes ────────────────────────────────────────────────────
_INST_COLORS = META_COLORS['instrument']
_DET_COLORS  = META_COLORS['detector']
_FRAG_COLORS = META_COLORS['fragmentation']
_ENZ_COLORS  = META_COLORS['enzyme']
_CHG_COLORS  = META_COLORS['charge']
_KING_COLORS = META_COLORS['kingdom']

# ── Stage definitions: (field, label, mapper, colour-dict) ───────────────────
TECH_STAGES = [
    ('instrument',    'Instrument',    _inst_fam,   _INST_COLORS),
    ('detector',      'Detector',      _detector_g, _DET_COLORS),
    ('fragmentation', 'Fragmentation', _ident,      _FRAG_COLORS),
]
BIO_STAGES = [
    ('enzyme',   'Enzyme',       _enzyme_g,   _ENZ_COLORS),
    ('charge',   'Charge state', _charge_g,   _CHG_COLORS),
    ('organism', 'Kingdom',      _to_kingdom, _KING_COLORS),
]

# ── Geometry helpers ─────────────────────────────────────────────────────────
def layout(counts, H=1.0, gap=GAP, mode=SCALE_MODE):
    transform = {'linear': lambda x: x, 'sqrt': lambda x: np.sqrt(x),
                 'log': lambda x: np.log1p(x)}[mode]
    t_vals = {k: transform(v) for k, v in counts.items()}
    t_sum  = sum(t_vals.values()) or 1
    avail  = H - gap * (len(counts) - 1)
    pos, cursor = {}, 0
    for k in counts.sort_values(ascending=False).index:
        h = (t_vals[k] / t_sum) * avail
        pos[k] = (cursor, cursor + h)
        cursor += h + gap
    return pos

def draw_ribbon(ax, x0, y0b, y0t, x1, y1b, y1t, color, alpha=ALPHA_RIB):
    xm = (x0 + x1) / 2
    verts = [(x0, y0b), (xm, y0b), (xm, y1b), (x1, y1b),
             (x1, y1t), (xm, y1t), (xm, y0t), (x0, y0t), (x0, y0b)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO,
             Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    ax.add_patch(patches.PathPatch(Path(verts, codes), facecolor=color,
                 edgecolor='none', alpha=alpha, zorder=2))

def draw_node(ax, x, yb, yt, color, w=NODE_W):
    ax.add_patch(patches.FancyBboxPatch(
        (x - w / 2, yb), w, max(yt - yb, 0.008), boxstyle='round,pad=0.004',
        facecolor=color, edgecolor='white', linewidth=1.5, zorder=5))

# ── Build per-stage node totals + adjacent joint flows ───────────────────────
def _stage_data(stages):
    n = len(stages)
    node  = [defaultdict(float) for _ in range(n)]
    joint = [defaultdict(lambda: defaultdict(float)) for _ in range(n - 1)]
    for _proj, _pd in _PROJ.items():
        R = _pd.get('runs', 0)
        gfrac = []
        for field, _lab, mapper, _col in stages:
            c = _pd.get(field, {}).get('counts', {})
            g = defaultdict(float)
            for k, v in c.items():
                gk = mapper(k)
                if gk is not None:
                    g[gk] += v
            tot = sum(g.values()) or 1
            gfrac.append({k: v / tot for k, v in g.items()})
        for s in range(n):
            for k, fr in gfrac[s].items():
                node[s][k] += R * fr
        for s in range(n - 1):
            for a, fa in gfrac[s].items():
                for b, fb in gfrac[s + 1].items():
                    joint[s][a][b] += R * fa * fb
    return node, joint

def _merge_small(node, joint):
    n = len(node)
    maps = []
    for s in range(n):
        maps.append({k: 'Other' for k, v in node[s].items() if v / _total * 100 < MIN_PCT})
    node2  = [defaultdict(float) for _ in range(n)]
    joint2 = [defaultdict(lambda: defaultdict(float)) for _ in range(n - 1)]
    for s in range(n):
        for k, v in node[s].items():
            node2[s][maps[s].get(k, k)] += v
    for s in range(n - 1):
        for a, row in joint[s].items():
            for b, v in row.items():
                joint2[s][maps[s].get(a, a)][maps[s + 1].get(b, b)] += v
    return node2, joint2

def draw_sankey(stages, name):
    node, joint = _merge_small(*_stage_data(stages))
    n   = len(stages)
    XS  = np.linspace(0.13, 0.87, n)
    ser = [pd.Series(dict(node[s])) for s in range(n)]
    pos = [layout(ser[s]) for s in range(n)]

    colmap = []
    for s, (_f, _lab, _mp, provided) in enumerate(stages):
        cats = list(ser[s].sort_values(ascending=False).index)
        cm = {}
        for idx, cat in enumerate(cats):
            cm[cat] = (provided or {}).get(cat) or ('#DEE2E6' if cat == 'Other'
                                                    else PALETTE[idx % len(PALETTE)])
        colmap.append(cm)

    fig, ax = plt.subplots(figsize=get_figsize(2))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')
    ax.set_xlim(0, 1); ax.set_ylim(-0.09, 1.12); ax.axis('off')

    for s in range(n - 1):
        srcc = {a: pos[s][a][0] for a in pos[s]}
        tgtc = {b: pos[s + 1][b][0] for b in pos[s + 1]}
        for a in ser[s].sort_values(ascending=False).index:
            for b in ser[s + 1].sort_values(ascending=False).index:
                nval = joint[s][a].get(b, 0)
                if nval < _total * 0.0008:
                    continue
                dha = (pos[s][a][1] - pos[s][a][0]) * nval / node[s][a]
                dhb = (pos[s + 1][b][1] - pos[s + 1][b][0]) * nval / node[s + 1][b]
                draw_ribbon(ax, XS[s], srcc[a], srcc[a] + dha,
                            XS[s + 1], tgtc[b], tgtc[b] + dhb, colmap[s][a])
                srcc[a] += dha; tgtc[b] += dhb

    for s in range(n):
        for cat, (yb, yt) in pos[s].items():
            draw_node(ax, XS[s], yb, yt, colmap[s][cat])
            pct = ser[s][cat] / _total * 100
            if s == 0:
                ax.text(XS[s] - NODE_W / 2 - 0.012, (yb + yt) / 2, f'{cat}  ({pct:.1f}%)',
                        ha='right', va='center', fontsize=8,
                        fontweight='bold' if pct > 10 else 'normal', color='#222')
            elif s == n - 1:
                ax.text(XS[s] + NODE_W / 2 + 0.012, (yb + yt) / 2, f'{cat}  ({pct:.1f}%)',
                        ha='left', va='center', fontsize=8,
                        fontweight='bold' if pct > 10 else 'normal', color='#222')
            else:
                ax.text(XS[s], (yb + yt) / 2, f'{cat}\n({pct:.1f}%)', ha='center',
                        va='center', fontsize=8, fontweight='bold', color='white', zorder=6)

    for x, (_f, lab, _mp, _col) in zip(XS, stages):
        ax.text(x, 1.07, lab, ha='center', fontsize=11, fontweight='bold', color='#333')
        ax.plot([x - 0.05, x + 0.05], [1.045, 1.045], color='#CCCCCC', lw=0.8)
    _scale = {'linear': 'linear scale', 'sqrt': '√-scale', 'log': 'log scale'}[SCALE_MODE]
    ax.text(0.5, -0.07, f'ribbon width ∝ runs ({_scale})', ha='center', fontsize=9, color='#888')

    plt.tight_layout()
    save_fig(fig, name)
    plt.show()

# ── Technical features: Instrument → Detector → Fragmentation ────────────────
draw_sankey(TECH_STAGES, 'sup_fig_sankey_technical')

# ── Biological features: Enzyme → Charge state → Kingdom ─────────────────────
draw_sankey(BIO_STAGES, 'sup_fig_sankey_biological')


In [ ]:
# ── Digestion enzyme donut ───────────────────────────────────
FIG_SIZE  = get_figsize(1)
TOP_N_ENZ = 7   # enzymes beyond this are grouped as "Other"

enz_raw   = pd.Series(LCMH['tiers']['lcfm']['enzyme']['counts']).sort_values(ascending=False)
enz_top   = enz_raw.head(TOP_N_ENZ).copy()
other_enz = enz_raw.iloc[TOP_N_ENZ:].sum()
if other_enz > 0:
    enz_top['Other'] = other_enz

fig, ax = plt.subplots(figsize=FIG_SIZE)
enz_colors = [META_COLORS['enzyme'].get(e, PALETTE[i % len(PALETTE)]) for i, e in enumerate(enz_top.index)]
wedges, texts, autotexts = ax.pie(
    enz_top.values,
    labels=None,                                      # enzyme names moved to the horizontal legend above
    autopct=lambda p: f'{p:.1f}%' if p >= 3 else '',  # suppress % on tiny slices (named in the legend)
    startangle=90,
    colors=enz_colors,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=1.5),
    pctdistance=0.78,
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
ax.text(0, 0, f'{fmt_n(enz_raw.sum())}\nspectra', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#333')
# ── Horizontal legend above the donut (enzyme + spectra count) ───────────
ax.legend(wedges, [f'{e} ({fmt_n(v)})' for e, v in zip(enz_top.index, enz_top.values)],
          loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=4,
          fontsize=9, frameon=False, columnspacing=1.2, handletextpad=0.5)
ax.set_aspect('equal')
fig.tight_layout()
save_fig(fig, 'fig_1g_enzyme_donut')
plt.show()


In [ ]:
# ── FIGURE PTM — Post-translational modifications by confidence tier ─────────
TOP_N    = 10
FIG_SIZE = get_figsize(2)

UNIMOD_NAMES = {
    # Common covalent modifications
    'UNIMOD:1':    'Acetyl (N-term)',
    'UNIMOD:3':    'Biotin (K)',
    'UNIMOD:4':    'Carbamidomethyl (C)',
    'UNIMOD:7':    'Deamidated (N/Q)',
    'UNIMOD:21':   'Phospho (S/T/Y)',
    'UNIMOD:27':   'Glu→pyroGlu (N-term E)',
    'UNIMOD:28':   'Pyro-Glu (N-term Q)',
    'UNIMOD:34':   'Methyl (K/R)',
    'UNIMOD:35':   'Oxidation (M)',
    'UNIMOD:36':   'Dimethyl (K/R)',
    'UNIMOD:37':   'Trimethyl (K)',
    'UNIMOD:43':   'HexNAc (N/S/T)',
    'UNIMOD:58':   'Propionyl (K)',
    'UNIMOD:64':   'Succinyl SA-label (N-term/K)',
    'UNIMOD:121':  'GlyGly / Ubiquitin (K)',
    'UNIMOD:122':  'Formyl (K/N-term)',
    # Isotope labels
    'UNIMOD:188':  'SILAC Label:13C(6)',
    'UNIMOD:214':  'iTRAQ4plex (K)',
    'UNIMOD:259':  'SILAC Lys+8',
    'UNIMOD:267':  'SILAC Arg+10',
    'UNIMOD:312':  'SILAC Arg+6',
    'UNIMOD:481':  'SILAC Lys+4',
    'UNIMOD:737':  'TMT6plex (K)',
    'UNIMOD:2016': 'TMTpro (K)',
    # Acylations (lysine PTMs)
    'UNIMOD:747':  'Malonyl (K)',
    'UNIMOD:1289': 'Butyryl (K)',
    'UNIMOD:1363': 'Crotonyl (K)',
    'UNIMOD:1848': 'Glutaryl (K)',
    'UNIMOD:1849': 'Hydroxyisobutyryl (K)',
    # N-linked glycans (complex)
    'UNIMOD:305':  'N-glycan G0F',
    'UNIMOD:308':  'N-glycan G2F',
    'UNIMOD:311':  'N-glycan G2',
    'UNIMOD:1408': 'N-glycan A2G2S2',
    'UNIMOD:1409': 'N-glycan A2G2S1',
    'UNIMOD:1410': 'N-glycan FA2G2S1',
    # N-linked glycans (high-mannose)
    'UNIMOD:137':  'N-glycan Man5',
    'UNIMOD:1465': 'N-glycan Man6',
    'UNIMOD:1480': 'N-glycan Man7',
    'UNIMOD:1504': 'N-glycan Man8',
    'UNIMOD:1531': 'N-glycan Man9',
    # N/O-linked glycans (hybrid/other)
    'UNIMOD:1761': 'Glycan dHex(1)Hex(3)HexNAc(2)',
    'UNIMOD:1775': 'Glycan dHex(1)Hex(3)HexNAc(5)',
}

# Rank PTMs by spectral occurrences in the LCFM tier
_lcfm_counts = LCMH['tiers']['lcfm']['modifications']['counts']
_top_ids = [uid for uid, _ in sorted(_lcfm_counts.items(), key=lambda x: -x[1])[:TOP_N]]
_labels  = [UNIMOD_NAMES.get(uid, uid) for uid in _top_ids]

# Per-PTM LCFM counts
_counts = np.array([_lcfm_counts.get(uid, 0) for uid in _top_ids], dtype=float)
_counts = np.where(_counts == 0, 0.5, _counts)   # avoid log(0) if any zero

# ── Draw ─────────────────────────────────────────────────────────
set_publication_style()
fig, ax = plt.subplots(figsize=FIG_SIZE)

_BAR_H = 0.6
_Y     = np.arange(TOP_N)

ax.barh(_Y, _counts, height=_BAR_H,
        color=TIER_PAL['LCFM'], label='LCFM', edgecolor='white', linewidth=0.5, zorder=3)

ax.set_xscale('log')
ax.set_xlabel('Spectral occurrences (log₁₀ scale)')
ax.set_yticks(_Y)
ax.set_yticklabels(_labels, fontsize=8.5)
ax.invert_yaxis()
ax.xaxis.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.5, zorder=0)
ax.set_axisbelow(True)

_handles = [mpatches.Patch(color=TIER_PAL['LCFM'], label='LCFM', edgecolor='grey', linewidth=0.4)]
ax.legend(handles=_handles, loc='lower right', frameon=True, fontsize=9, framealpha=0.9)

fig.tight_layout()
save_fig(fig, 'fig_1i_ptm_distribution')
plt.show()


---
# Part 4 — LCFM feature distributions
Spectrum-level and across-project distributions for the LCFM tier.

In [ ]:
# ── LCFM peptide-length distribution (spectrum level) ────────────────────────
# The only genuine per-spectrum histogram in the stats file: collision energy,
# precursor m/z and the confidence scores are stored as summary statistics only,
# so no true distribution can be reconstructed for them at tier level.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

_pl   = LCMH['tiers']['lcfm']['peptide_length']
_bins = {int(k): v for k, v in _pl['histogram'].items()}
_x    = np.array(sorted(_bins))
_y    = np.array([_bins[k] for k in _x], dtype=float)
_pct  = _y / _y.sum() * 100

fig, ax = plt.subplots(figsize=get_figsize(1))
ax.axvspan(_pl['mean'] - _pl['std'], _pl['mean'] + _pl['std'],
           color=TIER_PAL['HCFM'], alpha=0.10, zorder=1, linewidth=0)
ax.bar(_x, _pct, width=0.86, color=TIER_PAL['LCFM'],
       edgecolor='white', linewidth=0.4, zorder=3)
ax.axvline(_pl['mean'], color=TIER_PAL['HCFM'], linewidth=1.4,
           linestyle='--', zorder=4)
ax.annotate(f"mean {_pl['mean']:.1f} ± {_pl['std']:.1f} aa",
            xy=(_pl['mean'], ax.get_ylim()[1]), xytext=(4, -6),
            textcoords='offset points', ha='left', va='top',
            fontsize=9.5, color=TIER_PAL['HCFM'], fontweight='bold')

ax.set_xlabel('Peptide length (aa)')
ax.set_ylabel('Spectra (%)')
ax.set_xlim(_x.min() - 0.8, _x.max() + 0.8)
ax.set_xticks(np.arange(10, _x.max() + 1, 5))
ax.text(0.99, 0.95, f'LCFM · {fmt_n(_y.sum())} spectra', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='#666')
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, 'sup_fig_peptide_length')
plt.show()

---
# Supplementary — feature comparison across confidence tiers
Precursor charge, collision energy and hyperscore for LCFM / MCFM / HCFM.

In [ ]:
# ── Precursor charge across the three tiers ──────────────────────────────────
# charge.counts is a real per-spectrum tally, so this is a true distribution.
# Counts, not percentages, as requested; log y because the tiers themselves
# differ ~10x and ~5x while charge states span five orders of magnitude, so a
# linear axis would leave everything but LCFM z=2/3 on the baseline.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

_CHG_ORDER = ['1', '2', '3', '4', '5+']

def _charge_bucket(counts):
    """Fold 5,6,7,... into '5+' and drop 0 (unassigned charge)."""
    _out = {k: 0 for k in _CHG_ORDER}
    _unknown = 0
    for _k, _v in counts.items():
        _z = int(_k)
        if _z <= 0:
            _unknown += _v
        elif _z <= 4:
            _out[str(_z)] += _v
        else:
            _out['5+'] += _v
    return _out, _unknown

_by_tier = {t: _charge_bucket(LCMH['tiers'][t.lower()]['charge']['counts'])
            for t in TIER_ORDER}

_x  = np.arange(len(_CHG_ORDER))
_bw = 0.26

fig, ax = plt.subplots(figsize=get_figsize(2))
for _i, _t in enumerate(TIER_ORDER):
    _vals = [_by_tier[_t][0][_c] for _c in _CHG_ORDER]
    ax.bar(_x + (_i - 1) * _bw, _vals, width=_bw, label=_t,
           color=TIER_PAL[_t], edgecolor='white', linewidth=0.8, zorder=3)

ax.set_yscale('log', base=10)
ax.set_xticks(_x)
ax.set_xticklabels(_CHG_ORDER)
ax.set_xlabel('Precursor charge state')
ax.set_ylabel('Spectra (log₁₀ scale)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda _y, _: fmt_n(_y)))
ax.legend(title='Tier', loc='upper right', fontsize=10)
_unk = sum(_by_tier[_t][1] for _t in TIER_ORDER)
ax.text(0.01, 0.97, f'unassigned charge excluded ({fmt_n(_unk)} spectra)',
        transform=ax.transAxes, ha='left', va='top', fontsize=8.5, color='#888')
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, 'sup_fig_precursor_charge_by_tier')
plt.show()

In [ ]:
# ── Collision energy and hyperscore across the three tiers ───────────────────
# Neither field carries a histogram anywhere in the stats files (0 of 82 projects,
# all tiers) — only count/mean/std/min/max — so a true distribution cannot be
# reconstructed. Shown as mean ± 1 SD over the observed range, which is what the
# data supports, with the spectra count behind each row stated on the axis.
set_publication_style()
plt.rcParams.update({'axes.grid': False})

_SPREADS = [
    ('sup_fig_collision_energy_by_tier', 'Collision energy',
     lambda _t: LCMH['tiers'][_t.lower()]['collision_energy']),
    ('sup_fig_hyperscore_by_tier',       'Hyperscore',
     lambda _t: LCMH['tiers'][_t.lower()]['confidence_scores']['hyperscore']),
]

for _name, _label, _get in _SPREADS:
    _stats = {t: _get(t) for t in TIER_ORDER}
    _ys    = np.arange(len(TIER_ORDER))[::-1]

    fig, ax = plt.subplots(figsize=get_figsize(2))
    for _y, _t in zip(_ys, TIER_ORDER):
        _s = _stats[_t]
        _mu, _sd, _lo, _hi = _s['mean'], _s['std'], _s['min'], _s['max']
        ax.hlines(_y, _lo, _hi, color=TIER_PAL['MCFM'], linewidth=1.3, zorder=2)
        for _x in (_lo, _hi):
            ax.vlines(_x, _y - 0.10, _y + 0.10,
                      color=TIER_PAL['MCFM'], linewidth=1.3, zorder=2)
        ax.add_patch(patches.Rectangle((_mu - _sd, _y - 0.19), 2 * _sd, 0.38,
                                       facecolor=TIER_PAL[_t], edgecolor='white',
                                       linewidth=1.0, zorder=3))
        ax.plot([_mu], [_y], marker='o', markersize=7, color=TIER_PAL['HCFM'],
                markeredgecolor='white', markeredgewidth=1.2, zorder=4)
        ax.text(_mu, _y + 0.26, f'{_mu:,.1f} ± {_sd:,.1f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=TIER_PAL['HCFM'])

    ax.set_yticks(_ys)
    ax.set_yticklabels([f'{t}\n({fmt_n(_stats[t]["count"])} spectra)' for t in TIER_ORDER],
                       fontsize=9)
    ax.set_ylim(-0.6, len(TIER_ORDER) - 0.35)
    ax.set_xlabel(_label)
    ax.tick_params(axis='y', length=0)
    ax.text(0.99, 0.97, 'bar = mean ± 1 SD · line = observed range',
            transform=ax.transAxes, ha='right', va='top', fontsize=8.5, color='#888')
    sns.despine(ax=ax, left=True)
    fig.tight_layout()
    save_fig(fig, _name)
    plt.show()